# Lekcja 03 - Agentowe Wzorce Projektowe

W tej lekcji poznajemy trzy podstawowe wzorce projektowe do tworzenia skutecznych agentów AI:

1. **Jasne instrukcje dla agenta** — Tworzenie precyzyjnych, definiujących rolę wskazówek, które kierują zachowaniem agenta  
2. **Strukturalny wynik z modelami Pydantic** — Zapewnianie, że agenci zwracają przewidywalne, zwalidowane dane  
3. **Agenci o pojedynczej odpowiedzialności** — Projektowanie skoncentrowanych agentów, z których każdy robi jedną rzecz dobrze  

Zastosujemy każdy wzorzec w scenariuszu **rekomendatora miejsc podróży**, stopniowo budując system, który może sugerować miejsca, sprawdzać dostępność i obsługiwać logistykę.


## Instalacja


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Wzorzec 1: Jasne instrukcje dla agenta

Najbardziej efektywny wzorzec jest również najprostszy: pisanie jasnych, szczegółowych instrukcji dla Twojego agenta.

Dobre instrukcje definiują:
- **Kim** jest agent (osobowość i ton)
- **Co** powinien robić (krok po kroku obowiązki)
- **Jak** powinien się zachowywać (ograniczenia i styl)

Poniżej tworzymy agenta concierge podróży z wyraźnymi instrukcjami, które kształtują każdą wygenerowaną przez niego odpowiedź.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Wzorzec 2: Strukturalny output z użyciem modeli Pydantic

Tekst swobodny jest przydatny w rozmowie, ale systemy dalszego przetwarzania potrzebują danych strukturalnych.  
Łącząc **modele Pydantic** z **funkcją narzędziową**, możemy:

- Zdefiniować dokładny schemat outputu agenta  
- Automatycznie weryfikować odpowiedzi  
- Niezawodnie integrować wyniki agenta z logiką aplikacji  

Wprowadzamy także narzędzie, które zwraca szczegóły destynacji, aby agent mógł oprzeć swoje rekomendacje na rzeczywistych danych.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Wzorzec 3: Agenci o Pojedynczej Odpowiedzialności

Złożone zadania zyskują na rozdzieleniu pracy między wielu skoncentrowanych agentów, z których każdy ma pojedynczą odpowiedzialność:

- **Ekspert od Miejsc Docelowych**, który zna miejsca i ich dostępność
- **Planista Logistyki**, który zajmuje się lotami, hotelami i planami podróży

To odzwierciedla zasadę inżynierii oprogramowania *separacji obowiązków* — każdego agenta łatwiej jest testować, utrzymywać i ulepszać niezależnie.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Podsumowanie

W tej lekcji zastosowaliśmy trzy wzorce projektowe oparte na agentach w scenariuszu rekomendacji podróży:

| Wzorzec | Główna idea | Korzyść |
|---|---|---|
| **Jasne instrukcje** | Określ personę, obowiązki i ograniczenia na początku | Spójne, zgodne z marką zachowanie agenta |
| **Strukturalne wyjście** | Używaj modeli Pydantic jako formatu odpowiedzi | Zweryfikowane, czytelne dla maszyn wyniki |
| **Pojedyncza odpowiedzialność** | Przydziel każdemu agentowi jedno skoncentrowane zadanie | Łatwiejsze testowanie, utrzymanie i komponowanie |

Te wzorce naturalnie się łączą — możesz połączyć jasne instrukcje ze strukturalnym wyjściem w agencie o pojedynczej odpowiedzialności, aby zbudować solidne, gotowe do produkcji systemy.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:  
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mimo że staramy się zapewnić dokładność, prosimy pamiętać, że tłumaczenia automatyczne mogą zawierać błędy lub nieścisłości. Oryginalny dokument w języku źródłowym powinien być uznawany za źródło autorytatywne. W przypadku informacji krytycznych zaleca się korzystanie z profesjonalnego, ludzkiego tłumaczenia. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z korzystania z tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
